<div style="background:#1C3257;color:#F7F3EB;padding:22px 26px;border-radius:10px;font-family:Calibri,Arial,sans-serif"><div style="color:#E08A6E;font-size:12px;letter-spacing:2px;font-weight:bold">MINERÍA DE DATOS · UNIDAD 2 · SEMANA 5 — SESIÓN 2 · UPCh 2026A</div><div style="font-size:26px;font-weight:bold;margin-top:6px">Lab 3 — BM25 y evaluación de búsqueda</div><div style="font-style:italic;color:#C9D4E4;margin-top:8px">Mejor ranking que TF-IDF, y cómo medirlo con métricas de IR</div></div>

## Reglas de entrega

- **Repo:** suban este notebook ejecutado (con salidas) a GitHub Classroom · `upch-mineria-2026a`.
- **`AI_USAGE.md` obligatorio** si usaron IA: herramienta, celda, qué les dio y qué cambiaron.
- **Defensa oral (eliminatoria):** se les preguntará por cualquier celda. Si no la pueden explicar, no hay calificación.
- **Tardías:** 25% (<24 h), 50% (<48 h), rechazado (>48 h).
- Lo evaluado son las celdas `# TODO` y las preguntas en **negritas**. El resto es andamiaje ya resuelto.
- **Sin librerías de NLP/IR para resolver:** TF-IDF, BM25, las métricas y K-Means van **desde cero**. `scikit-learn` solo se permite donde se indique *verificación*.


## Objetivo

Dos partes. **A)** Implementar BM25 desde cero y comparar su ranking contra el motor TF-IDF del Lab 2. **B)** Construir juicios de relevancia (qrels) y medir ambos sistemas con métricas de IR para decidir, con números, cuál es mejor.


## 0 · Corpus procesado del Lab 1

In [5]:
import json, math, re, unicodedata
from collections import Counter
import spacy
try:
    nlp = spacy.load('es_core_news_sm')
except OSError:
    from spacy.cli import download; download('es_core_news_sm')
    nlp = spacy.load('es_core_news_sm')


with open('corpus_procesado.json', encoding='utf-8') as fh:
    corpus = json.load(fh)              # del Lab 1
documentos = [d['tokens'] for d in corpus]
ids = [d['id'] for d in corpus]
print(f'{len(corpus)} documentos. Ejemplo {ids[0]}:', documentos[0][:8])

14 documentos. Ejemplo d01: ['fuerte', 'lluvia', 'provocar', 'inundación', 'colonia', 'sur', 'tuxtla', 'gutierrez']


### Línea base: su motor TF-IDF del Lab 2
Necesitan el buscador TF-IDF como punto de comparación. Peguen sus funciones del Lab 2.

In [7]:
# TODO: peguen sus funciones del Lab 2: tf, idf, tfidf, coseno, preprocesar,
#       y construyan IDF, INDICE, vectorizar_consulta y buscar_tfidf(q, k=5).

RE_URL  = re.compile(r'https?://\S+')
RE_HTML = re.compile(r'<[^>]+>')
RE_EMOJI = re.compile('[\U0001F300-\U0001FAFF\u2600-\u27BF]')

stopwords_es = nlp.Defaults.stop_words
CONSERVAR = {'no', 'sin'}   # palabras que NO quieren tratar como stopword (con criterio de dominio)
MIS_STOPWORDS = set(stopwords_es) - CONSERVAR

def normalizar(texto, quitar_acentos=True):
    """Devuelve el texto normalizado (aun como string, sin tokenizar)."""
    texto = unicodedata.normalize('NFC', texto)
    texto = texto.lower()
    texto = RE_HTML.sub(' ', texto)
    texto = RE_URL.sub(' ', texto)
    if quitar_acentos:
        texto = unicodedata.normalize('NFD', texto)
        texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

def preprocesar(texto):
    texto_norm = normalizar(texto, quitar_acentos=True)
    doc = nlp(texto_norm)
    tokens = []
    for token in doc:
        if token.is_punct or token.is_space:
            continue
        lemma = token.lemma_.lower()
        if lemma in MIS_STOPWORDS:
            continue
        if len(lemma) <= 1 and not lemma.isdigit():
            continue
        tokens.append(lemma)
    return tokens

def tf(doc):
    """doc: lista de terminos -> dict termino:tf (frecuencia relativa)."""
    total = len(doc)
    if total == 0:
        return {}
    counts = Counter(doc)
    return {term: count / total for term, count in counts.items()}

def idf(corpus):
    """corpus: lista de docs (cada uno lista de terminos) -> dict termino:idf."""
    n_docs = len(corpus)
    df = Counter()
    for doc in corpus:
        for term in set(doc):  # set() asegura contar el doc una sola vez por término
            df[term] += 1
    return {term: math.log(n_docs / df_t) for term, df_t in df.items()}

def tfidf(doc, idf_):
    """-> dict termino:peso tf-idf para un documento."""
    tf_doc = tf(doc)
    return {term: tf_val * idf_.get(term, 0) for term, tf_val in tf_doc.items()}

# Construir el indice: un vector tf-idf (dict) por documento
IDF = idf(documentos)
INDICE = [tfidf(doc, IDF) for doc in documentos]

# Sanity check: el termino mas pesado de d04 (sequia/maiz) deberia ser tematico
import operator
top = sorted(INDICE[3].items(), key=operator.itemgetter(1), reverse=True)[:5]
print('Terminos top de', corpus[3]['id'], '->', top)

def vectorizar_consulta(texto):
    """texto libre -> vector tf-idf (dict) usando el IDF del corpus."""
    texto_proc = preprocesar(texto)
    return tfidf(texto_proc, IDF)

print(vectorizar_consulta('sequia en los cultivos'))

def coseno(v1, v2):
    """Similitud coseno entre dos vectores dispersos (dicts). Maneja el caso de norma 0."""
    # Producto punto: solo recorremos las claves de v1 (las que no estan en v2 aportan 0)
    dot = sum(peso * v2.get(term, 0) for term, peso in v1.items())

    norma1 = math.sqrt(sum(peso ** 2 for peso in v1.values()))
    norma2 = math.sqrt(sum(peso ** 2 for peso in v2.values()))

    if norma1 == 0 or norma2 == 0:
        return 0.0

    return dot / (norma1 * norma2)

def buscar_tfidf(q, k=5):
    """Devuelve los k documentos mas relevantes como (id, titulo, score)."""
    q = vectorizar_consulta(q)

    resultados = [
        (doc['id'], doc['titulo'], coseno(q, vector))   
        for doc, vector in zip(corpus, INDICE)
    ]

    resultados.sort(key=lambda x: x[2], reverse=True)
    return resultados[:k]

Terminos top de d04 -> [('gravemente', 0.16494108310095365), ('cultivo', 0.16494108310095365), ('maiz', 0.16494108310095365), ('frijol', 0.16494108310095365), ('fronterizo', 0.16494108310095365)]
{'sequia': 0.9729550745276566, 'cultivo': 1.3195286648076292}


---
## Parte A · BM25 desde cero

**A.1** IDF de BM25 (variante suavizada, nunca negativa) y la función `bm25`. Sigan la fórmula de la clase:
$$\text{score}(d,q)=\sum_{t\in q}\text{IDF}(t)\cdot\frac{f\,(k_1+1)}{f+k_1\,(1-b+b\,|d|/\text{avgdl})}$$

In [8]:
avgdl = sum(len(doc) for doc in documentos) / len(documentos)

def idf_bm25(corpus):
    # TODO: ln(1 + (N - df + 0.5)/(df + 0.5))  por termino
    n_docs = len(corpus)
    df = Counter()
    for doc in corpus:
        for term in set(doc):
            df[term] += 1
    return {
        term: math.log(1 + (n_docs - df_t + 0.5) / (df_t + 0.5))
        for term, df_t in df.items()
    }


def bm25(doc, q, k1=1.5, b=0.75):
    # TODO: acumular el score termino por termino segun la formula
    counts = Counter(doc)
    dl = len(doc)
    score = 0.0
    for term in q:
        f = counts.get(term, 0)
        if f == 0:
            continue
        idf_t = IDF_BM25.get(term, 0)
        numerador = f * (k1 + 1)
        denominador = f + k1 * (1 - b + b * (dl / avgdl))
        score += idf_t * (numerador / denominador)
    return score

# Construir IDF de BM25 sobre todo el corpus (una sola vez)
IDF_BM25 = idf_bm25(documentos)

**A.2** Buscador BM25, análogo a `buscar_tfidf` pero ranqueando por score BM25.

In [ ]:
def buscar_bm25(consulta, k=5, k1=1.5, b=0.75):
    # TODO: preprocesar la consulta -> score bm25 contra cada doc -> top-k
    raise NotImplementedError

**A.3** Comparación lado a lado. Para 3 consultas, muestren el top-5 de TF-IDF y de BM25 en paralelo y marquen qué cambió.

In [ ]:
# TODO: 3 consultas; imprimir top-5 de TF-IDF y BM25 lado a lado y senalar diferencias.
raise NotImplementedError

**A.4** Barrido de parámetros. Observen cómo se mueve el ranking de una consulta al variar (k₁, b).

In [ ]:
# TODO: barrer (k1, b) in {1.2, 2.0} x {0.0, 0.75, 1.0} para una consulta
#       e imprimir el top-3 en cada caso.
raise NotImplementedError

_Describan cómo y por qué cambia el ranking al mover k₁ y b:_ 

---
## Parte B · Evaluación con métricas de IR

**B.1** Juicios de relevancia (qrels). Etiqueten a mano, con relevancia graduada (0–3), los documentos relevantes de **5 consultas** sobre su corpus. Completen el diccionario.

In [ ]:
# relevancia graduada: 3 = muy relevante, 2 = relevante, 1 = marginal, ausente = 0
qrels = {
    'sequia y cultivos':  {'d04': 3, 'd02': 2},   # ejemplo dado
    # TODO: agreguen 4 consultas mas, etiquetadas a mano. Justifiquen sus juicios.
}
assert len(qrels) >= 5, 'Faltan consultas por etiquetar'

**B.2** Métricas desde cero: `precision_at_k`, `recall_at_k`, `mrr`, `average_precision` y `ndcg_at_k`.

In [ ]:
def _rel(qid, doc): return qrels[qid].get(doc, 0)
def _relevantes(qid): return {d for d, g in qrels[qid].items() if g > 0}

def precision_at_k(ranking, qid, k=5):
    # TODO
    raise NotImplementedError
def recall_at_k(ranking, qid, k=5):
    # TODO
    raise NotImplementedError
def mrr(ranking, qid):
    # TODO: 1/(posicion del primer relevante)
    raise NotImplementedError
def average_precision(ranking, qid):
    # TODO: promedio de la precision en cada acierto
    raise NotImplementedError
def ndcg_at_k(ranking, qid, k=5):
    # TODO: DCG/IDCG con relevancia graduada
    raise NotImplementedError

**B.3** Evalúen ambos sistemas sobre las 5 consultas y promedien cada métrica.

In [ ]:
# TODO: funcion evaluar(buscar_fn) que promedie las 5 metricas sobre los qrels;
#       construyan la tabla comparativa TF-IDF vs BM25 con la columna de mejora.
raise NotImplementedError

**B.4** ¿Qué sistema y qué (k₁, b) eligen para producción, y **con qué métrica** lo justifican?

_Su decisión:_ 

## Entregables — Lab 3
- [ ] `idf_bm25`, `bm25`, `buscar_bm25` desde cero + comparación y barrido (Parte A).
- [ ] `qrels` de 5 consultas + las 5 métricas desde cero (Parte B).
- [ ] Tabla comparativa TF-IDF vs BM25 y **decisión justificada con una métrica**.
- [ ] `AI_USAGE.md` actualizado si usaron IA.
